In [13]:
import pandas as pd

columns = [
    "checking_account_status",
    "duration",
    "credit_history",
    "purpose",
    "credit_amount",
    "savings_account",
    "employment_since",
    "installment_rate",
    "personal_status_sex",
    "other_debtors",
    "residence_since",
    "property",
    "age",
    "other_installment_plans",
    "housing",
    "existing_credits",
    "job",
    "dependents",
    "telephone",
    "foreign_worker",
    "credit_risk"
]

df = pd.read_csv(
    "../data/german.data",
    sep=" ",
    header=None,
    names=columns
)

In [14]:
df["credit_risk"].value_counts(dropna=False)

credit_risk
1    700
2    300
Name: count, dtype: int64

In [15]:
df["credit_risk"] = df["credit_risk"].map({1: 0, 2: 1})
df["credit_risk"].value_counts(dropna=False)

credit_risk
0    700
1    300
Name: count, dtype: int64

In [16]:
X = df.drop("credit_risk", axis=1)
y = df["credit_risk"]

print(y.value_counts(dropna=False))
print(y.isna().sum())

credit_risk
0    700
1    300
Name: count, dtype: int64
0


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (800, 20)
Testing data: (200, 20)


In [18]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

print("Numeric features:", numeric_features)
print()
print("Categorical features:", categorical_features)

Numeric features: Index(['duration', 'credit_amount', 'installment_rate', 'residence_since',
       'age', 'existing_credits', 'dependents'],
      dtype='object')

Categorical features: Index(['checking_account_status', 'credit_history', 'purpose',
       'savings_account', 'employment_since', 'personal_status_sex',
       'other_debtors', 'property', 'other_installment_plans', 'housing',
       'job', 'telephone', 'foreign_worker'],
      dtype='object')


In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [20]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['duration', 'credit_amount', 'installment_rate', 'residence_since',
       'age', 'existing_credits', 'dependents'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='mos...quent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['checking_account_status', 'credit_history', 'purpose',
       'savings_account', 'employment_since', 'personal_status_sex',
       'other_debtors', 'property', 'other_installment_plans', 'housing',
       'job', 'telephone', 'foreign_worker'],
      dtype='object'))])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [21]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.82      0.89      0.85       140
           1       0.67      0.53      0.59        60

    accuracy                           0.78       200
   macro avg       0.74      0.71      0.72       200
weighted avg       0.77      0.78      0.77       200

[[124  16]
 [ 28  32]]
ROC-AUC: 0.8040476190476191


In [22]:
import joblib

joblib.dump(model, "../models/loan_default_model.pkl")
print("Model saved successfully.")

Model saved successfully.
